In [59]:
# If needed, install dependencies
!pip install transformers torch pandas tqdm

import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm


In [60]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [61]:
REVIEWS_FILE = "/content/sample_data/reviews-2.csv"
LISTINGS_FILE = "/content/sample_data/listings-summary.csv"
MODEL_NAME = "nlptown/bert-base-multilingual-uncased-sentiment"
MAX_REVIEWS = None
MIN_TEXT_LENGTH = 10

In [62]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(device)
model.eval()


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(105879, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1

In [63]:
import os
print("Current working directory:", os.getcwd())
print("Files in sample_data:", os.listdir("/content/sample_data"))


Current working directory: /content
Files in sample_data: ['README.md', 'anscombe.json', 'reviews-2.csv', '.ipynb_checkpoints', 'sentiment_by_neighbourhood.csv', 'listings-summary.csv', 'california_housing_test.csv', 'california_housing_train.csv', 'mnist_test.csv', 'mnist_train_small.csv']


In [64]:
usecols_reviews = ['listing_id', 'comments']
usecols_listings = ['id', 'neighbourhood']

reviews_df = pd.read_csv(
    REVIEWS_FILE,
    usecols=["listing_id", "comments"],
    quotechar='"',
    escapechar='\\',
    encoding='utf-8',
    on_bad_lines='skip'  # pandas >= 1.3
)

listings_df = pd.read_csv(
    LISTINGS_FILE,
    usecols=["id", "neighbourhood"],
    sep=';',                     # <-- FIXED
    quotechar='"',
    escapechar='\\',
    encoding='utf-8',
    on_bad_lines='skip'
)

if MAX_REVIEWS:
    reviews_df = reviews_df.head(MAX_REVIEWS)

print(f"✅ Loaded {len(reviews_df)} reviews")
reviews_df.head(3)


✅ Loaded 388571 reviews


,listing_id,comments
0,63413,Super séjour. Paola vous dira tout ce qu'il fa...
1,63413,We had a beautiful stay in Napels! Paola and h...
2,63413,Accueil chaleureux. Appartement est bien placé...


In [65]:
def get_sentiment_scores_batch(texts):
    inputs = tokenizer(texts, return_tensors="pt", truncation=True, padding=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits
    probs = F.softmax(logits, dim=1)

    star_ratings = torch.arange(1, 6, dtype=torch.float).to(device)
    scores = torch.sum(probs * star_ratings, dim=1)
    return scores.cpu().tolist()


In [68]:
import re
from tqdm import tqdm

batch_size = 64
results = []
skipped_batches = 0
total_skipped_reviews = 0

def strip_html_tags(text):
    return re.sub(r'<[^>]+>', '', text)

# Clean comments: remove HTML tags, fillna, convert to str, strip whitespace
reviews_df['comments'] = (
    reviews_df['comments']
    .fillna("")
    .astype(str)
    .apply(strip_html_tags)
    .str.strip()
)

print(f"Starting batch inference over {len(reviews_df)} reviews...")

for i in tqdm(range(0, len(reviews_df), batch_size)):
    batch_df = reviews_df.iloc[i:i + batch_size]

    # Filter out very short reviews (<10 chars)
    batch_df = batch_df[batch_df['comments'].str.len() >= 10]

    if batch_df.empty:
        continue

    batch_texts = batch_df['comments'].tolist()
    batch_ids = batch_df['listing_id'].tolist()

    try:
        scores_list = get_sentiment_scores_batch(batch_texts)  # List of floats
    except Exception as e:
        skipped_batches += 1
        total_skipped_reviews += len(batch_df)
        print(f"Warning: Batch {i}-{i+batch_size} failed with error: {e}")
        continue

    # Save scores directly — no indexing into probs
    for listing_id, score in zip(batch_ids, scores_list):
        results.append({"listing_id": listing_id, "compound": score})

# Convert to DataFrame
sentiment_df = pd.DataFrame(results)

print("\nFinished sentiment analysis.")
print(f"Total reviews in file: {len(reviews_df)}")
print(f"Reviews processed successfully: {len(sentiment_df)}")
print(f"Skipped batches: {skipped_batches}")
print(f"Skipped reviews (from failed batches or short): {total_skipped_reviews}")


Starting batch inference over 388571 reviews...


100%|██████████| 6072/6072 [42:34<00:00,  2.38it/s]



Finished sentiment analysis.
Total reviews in file: 388571
Reviews processed successfully: 384249
Skipped batches: 0
Skipped reviews (from failed batches or short): 0


In [69]:
merged = pd.merge(
    sentiment_df,
    listings_df,
    left_on="listing_id",
    right_on="id",
    how="left"
)

merged = merged.dropna(subset=["neighbourhood"])
merged = merged[["listing_id", "compound", "neighbourhood"]]
merged.head()

,listing_id,compound,neighbourhood
0,63413,4.648525,Chiaia
1,63413,4.816487,Chiaia
2,63413,4.174839,Chiaia
3,63413,4.984756,Chiaia
4,63413,4.489517,Chiaia


In [70]:
grouped = merged.groupby("neighbourhood").agg(
    avg_sentiment=("compound", "mean"),
    review_count=("compound", "count")
).reset_index()

grouped.sort_values("avg_sentiment", ascending=False).head(10)


,neighbourhood,avg_sentiment,review_count
22,Scampia,4.570561,26
11,Piscinola,4.537826,810
10,Pianura,4.530718,172
27,Vomero,4.529320,12441
6,Fuorigrotta,4.514434,3591
17,San Ferdinando,4.510085,37990
0,Arenella,4.508426,3844
15,Posillipo,4.504774,4063
19,San Giuseppe,4.502177,26757
4,Chiaia,4.498966,23866


In [71]:
OUTPUT_PATH = "/content/sample_data/sentiment_by_neighbourhood.csv"
grouped.to_csv(OUTPUT_PATH, index=False)
print(f"✅ Saved to {OUTPUT_PATH}")


✅ Saved to /content/sample_data/sentiment_by_neighbourhood.csv
